In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from pydantic_ai.models.openai import OpenAIResponsesModel
from pydantic_ai.providers.openai import OpenAIProvider
import asyncio
from pydantic_ai.mcp import MCPServerStdio
from datetime import datetime
from pydantic_ai import Tool, Agent


In [ ]:
# Define interface with model
provider = OpenAIProvider(
        base_url="http://127.0.0.1:44444/v1"
)
#Define class to interface with LLM server
agent_model = OpenAIResponsesModel("/scratch/general/vast/app-repo/huggingface/Qwen/Qwen3-8B", provider=provider)
agent = Agent(model=agent_model, end_strategy='exhaustive')

In [3]:
# Unfortunately running these calls in a Jupyter notebook is more complicated than a straight Python script
# so we must run asynchronously

# our async function to retrieve output
async def run_async(prompt: str) -> str:
    async with agent.run_mcp_servers():
        result = await agent.run(prompt)
        return result.output


In [5]:
print('baseline response', await run_async("What is the date today?"))

baseline response <think>
Okay, the user is asking for the current date. I need to figure out what today's date is. Let me check my system's current date. Wait, I should make sure I'm using the correct time zone. The user might be in a different location, but since they didn't specify, I'll assume their local time.

Hmm, I should confirm the current date. Let me think. If today is October 23rd, 2023, then that's the date. But I need to verify. Let me check a reliable source. Oh, right, I can't access external information, so I have to rely on my internal clock. 

Wait, maybe I should respond with the date in the format they might expect. They might want it in MM/DD/YYYY or DD/MM/YYYY. Since I don't know their region, I'll use the international standard, which is YYYY-MM-DD. That way, it's clear and avoids confusion. 

Also, I should mention the time zone, but since the user didn't specify, I'll just state the date. Let me double-check my internal date. Yes, today is October 23, 2023. S

In [6]:
# Define our tool to help out the AI with time
@Tool
async def get_current_date() -> str:
    """Return the current date/time as an ISO-formatted string."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


In [7]:
# Redefine our agent and tell it it has a tool
agent = Agent(
    model=agent_model,
    tools=[get_current_date],
    system_prompt = (
        "You have access to:\n"
        "   1. get_current_date()\n"
        "Use this tool for date/time questions."
    )
)


In [8]:
# Try again
print(await run_async("What is the date today?"))


<think>
Okay, the user asked for today's date. I called the get_current_date function, which returned the ISO-formatted date and time: 2026-05-06 11:40:38. The user probably just needs the date part, so I'll extract the date portion from the response. The answer should be straightforward, just the date. Let me make sure there are no time zones mentioned, but since the function returns the local time, I'll present it as is. The user might be checking if the date is correct or needs it for a specific purpose. I'll keep it simple and clear.
</think>

The current date is **2026-05-06**.
